# Character replacement on a free T4 — ComfyUI + Wan 2.2 Animate

Give it **a video** (someone moving) and **a picture of a character**. It swaps the person in the video for your
character and keeps the original background, motion, expressions and lighting. Wan 2.2 Animate's *replacement mode*
runs inside ComfyUI with no UI needed, so it works on Colab's free tier.

**How to use it:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells in order (▶). The first
run downloads about 18 GB of models (5–8 minutes).

| Step | What happens |
|---|---|
| 1 | Installs ComfyUI and the nodes the workflow needs |
| 2 | Settings: size, length, prompt, quality |
| 3 | Downloads the models (quantised to fit a T4) |
| 4 | Upload your video and character picture |
| 5 | Pick the person to replace (click-free: type a point, see it on the frame) |
| 6–7 | Builds the workflow and starts ComfyUI in the background |
| 8 | Finds the pose, face and person once, then renders 77-frame chunks (about 5 s each at 16 fps) that continue each other; re-running it after a crash resumes |
| 9 | Joins, adds the original sound, optional 30 fps / 1080×1920 upscale, download |

**Speed on a T4:** roughly 8–15 minutes per 77-frame chunk at 480×832. A 5-second clip is 1–2 chunks.

**Use it responsibly.** Only use footage of yourself or of people who agreed to it, and character pictures you made or
have the rights to. Don't put a real person's likeness into a video without their consent. Models: Wan 2.2 Animate
(Apache-2.0), SAM 2 (Apache-2.0), DWPose (Apache-2.0).

In [ ]:
#@title 1 · Install ComfyUI and the nodes (about 3 minutes)
import subprocess, os, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none — set Runtime → Change runtime type → T4 GPU, then run this again")
ROOT = "/content/ComfyUI"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise SystemExit(f"failed: {cmd}")
if not os.path.exists(ROOT):
    sh(f"git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI {ROOT}")
NODES = {
    "ComfyUI-GGUF": "city96/ComfyUI-GGUF",                       # quantised Wan + text encoder
    "ComfyUI-KJNodes": "kijai/ComfyUI-KJNodes",                  # resize, mask and crop helpers
    "ComfyUI-VideoHelperSuite": "Kosinkadink/ComfyUI-VideoHelperSuite",  # load / save video
    "comfyui_controlnet_aux": "Fannovel16/comfyui_controlnet_aux",       # DWPose (body, hands, face)
    "ComfyUI-segment-anything-2": "kijai/ComfyUI-segment-anything-2",    # SAM 2 to follow the person
    "ComfyUI-WanVideoWrapper": "kijai/ComfyUI-WanVideoWrapper",          # face mask from pose keypoints
}
for name, repo in NODES.items():
    if not os.path.exists(f"{ROOT}/custom_nodes/{name}"):
        sh(f"git clone -q --depth 1 https://github.com/{repo} {ROOT}/custom_nodes/{name}")
# keep Colab's own CUDA build of torch
reqs = [f"{ROOT}/requirements.txt"] + [f"{ROOT}/custom_nodes/{n}/requirements.txt" for n in NODES]
lines = set()
for f in reqs:
    if os.path.exists(f):
        for l in open(f):
            l = l.split("#")[0].strip()
            if l and not l.split("=")[0].split(">")[0].split("<")[0].strip().lower() in ("torch", "torchvision", "torchaudio"):
                lines.add(l)
open("/content/reqs.txt", "w").write("\n".join(sorted(lines)))
sh(f"{sys.executable} -m pip install -q -r /content/reqs.txt")
sh("apt-get -qq install -y ffmpeg > /dev/null")
print("✓ installed")

In [ ]:
#@title 2 · Settings
WIDTH = 480    #@param {type:"integer"}
HEIGHT = 832   #@param {type:"integer"}
#@markdown 480×832 is vertical 9:16 (reels). Use 832×480 for landscape. Keep both multiples of 16; bigger is much slower on a T4.
SECONDS = 5    #@param {type:"number"}
#@markdown How much of the video to replace (at 16 fps). Longer runs are made in 77-frame chunks that continue each other.
START_AT_SECONDS = 0  #@param {type:"number"}
CHUNK_FRAMES = 77     #@param {type:"integer"}
PROMPT = "a person in a navy hoodie moving naturally in a softly lit living room, realistic, detailed, natural light"  #@param {type:"string"}
#@markdown Describe your character and the scene in a few words. The picture does most of the work.
NEGATIVE = "blurry, low quality, distorted face, extra fingers, deformed hands, flicker, watermark, text, static frame"  #@param {type:"string"}
QUANT = "Q4_K_S"  #@param ["Q3_K_M", "Q4_K_S", "Q4_K_M", "Q5_K_M"]
#@markdown Q4_K_S fits the free Colab's 12.7 GB of RAM with room to spare. Q3_K_M if it crashes; Q4_K_M / Q5_K_M look a little better on Colab Pro (more RAM).
STEPS = 6   #@param {type:"slider", min:4, max:10, step:1}
#@markdown The speed LoRA makes 4–8 steps enough.
SEED = 7    #@param {type:"integer"}
RELIGHT_STRENGTH = 1.0  #@param {type:"slider", min:0, max:1, step:0.1}
#@markdown Relights your character to match the scene. 0 keeps the picture's own lighting.
MASK_GROW = 20  #@param {type:"integer"}
#@markdown How far the replaced area reaches past the person's outline (pixels). Raise it if bits of the original person show at the edges.
SMOOTH_30FPS_1080P = True  #@param {type:"boolean"}

SKIP_FIRST_FRAMES = int(round(START_AT_SECONDS * 16))
PERSON_INDEX = 0
SAM2_MODEL = "sam2.1_hiera_base_plus.safetensors"
UNET_FILE = f"Wan2.2-Animate-14B-{QUANT}.gguf"
TEXT_ENCODER_FILE = "umt5-xxl-encoder-Q5_K_M.gguf"
LIGHTX2V_FILE = "lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors"
RELIGHT_FILE = "wan2.2_animate_14B_relight_lora_bf16.safetensors"
VAE_FILE = "wan_2.1_vae.safetensors"
CLIP_VISION_FILE = "clip_vision_h.safetensors"
assert WIDTH % 16 == 0 and HEIGHT % 16 == 0, "WIDTH and HEIGHT must be multiples of 16"
CHUNK_FRAMES = max(9, (CHUNK_FRAMES - 1) // 4 * 4 + 1)
print(f"✓ {WIDTH}×{HEIGHT}, {SECONDS}s from {START_AT_SECONDS}s, {UNET_FILE}, {STEPS} steps")

In [ ]:
#@title 3 · Download the models (about 18 GB, first run only)
import os, shutil
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
M = f"{ROOT}/models"
FILES = [  # (repo, file in repo, ComfyUI folder)
    ("QuantStack/Wan2.2-Animate-14B-GGUF", UNET_FILE, "diffusion_models"),
    ("city96/umt5-xxl-encoder-gguf", TEXT_ENCODER_FILE, "text_encoders"),
    ("Kijai/WanVideo_comfy", f"Lightx2v/{LIGHTX2V_FILE}", "loras"),
    ("Comfy-Org/Wan-2.2", f"split_files/loras/{RELIGHT_FILE}", "loras"),
    ("Comfy-Org/Wan-2.1", f"split_files/vae/{VAE_FILE}", "vae"),
    ("Comfy-Org/Wan-2.1", f"split_files/clip_vision/{CLIP_VISION_FILE}", "clip_vision"),
]
def get(item):
    repo, f, folder = item
    dest = f"{M}/{folder}/{os.path.basename(f)}"
    if os.path.exists(dest): return f"already here: {os.path.basename(f)}"
    os.makedirs(f"{M}/{folder}", exist_ok=True)
    p = hf_hub_download(repo, f, local_dir=f"/content/hf/{folder}")
    shutil.move(p, dest)
    return f"✓ {os.path.basename(f)} ({os.path.getsize(dest) / 1e9:.1f} GB)"
with ThreadPoolExecutor(3) as ex:
    for r in ex.map(get, FILES): print(r)
shutil.rmtree("/content/hf", ignore_errors=True)
print("DWPose and SAM 2 download themselves (about 0.5 GB) on the first render.")

In [ ]:
#@title 4 · Upload your video and your character picture
#@markdown The video: one clearly visible person, steady-ish camera, a few seconds long. The picture: your character,
#@markdown full body or at least head to waist, plain background works best, roughly the same framing as the person in the video.
import os, shutil
from google.colab import files
IN = f"{ROOT}/input"; os.makedirs(IN, exist_ok=True)
print("Choose the VIDEO (mp4 / mov / webm):")
up = files.upload(); v = next(iter(up))
ext = os.path.splitext(v)[1].lower() or ".mp4"
VIDEO = f"source{ext}"; shutil.move(v, f"{IN}/{VIDEO}")
print("Choose the CHARACTER picture (png / jpg / webp):")
up = files.upload(); p = next(iter(up))
REF = "character" + (os.path.splitext(p)[1].lower() or ".png"); shutil.move(p, f"{IN}/{REF}")
print(f"✓ {VIDEO}, {REF}")

In [ ]:
#@title 5 · Pick the person to replace
POINTS = "240,400"  #@param {type:"string"}
#@markdown One or more points ON the person in the first frame, as x,y (in the grid below). Separate several points
#@markdown with spaces, e.g. `240,300 240,560` (head and body). Run this cell again after changing them.
import subprocess, numpy as np, matplotlib.pyplot as plt
from PIL import Image
frame = "/content/first_frame.png"
vf = f"scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT}"   # same crop as the workflow
subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-ss", str(START_AT_SECONDS), "-i", f"{ROOT}/input/{VIDEO}",
                "-frames:v", "1", "-vf", vf, frame], check=True)
POINTS_XY = [tuple(int(float(c)) for c in p.split(",")) for p in POINTS.split()]
img = np.asarray(Image.open(frame))
fig, axes = plt.subplots(1, 2, figsize=(10, 9 * HEIGHT / WIDTH / 2 + 1))
axes[0].imshow(img); axes[0].set_title("First frame (grid every 50 px)")
axes[0].set_xticks(range(0, WIDTH, 50)); axes[0].set_yticks(range(0, HEIGHT, 50)); axes[0].grid(alpha=.35)
axes[0].tick_params(labelsize=7)
for x, y in POINTS_XY: axes[0].plot(x, y, "o", ms=12, mfc="lime", mec="black")
axes[1].imshow(Image.open(f"{ROOT}/input/{REF}")); axes[1].set_title("Your character"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("Points:", POINTS_XY, "— the green dots should sit on the person you want replaced.")

In [ ]:
#@title 6 · The workflow (Wan 2.2 Animate, replacement mode) — no need to edit
#@markdown Two stages, to stay inside the free Colab's 12.7 GB of RAM:
#@markdown **A** finds the pose, the face and the person (SAM 2) once for the whole clip and saves them as small videos;
#@markdown **B** renders each chunk from just its own slice of those videos, so only ~77 frames are ever in memory.
import json

def _save(g, i, images, prefix, crf=8):
    g[str(i)] = {"class_type": "VHS_VideoCombine", "inputs": dict(
        images=images, frame_rate=16, loop_count=0, filename_prefix=prefix, format="video/h264-mp4", pix_fmt="yuv420p",
        crf=crf, save_metadata=False, trim_to_audio=False, pingpong=False, save_output=True)}

def _load(g, i, video, skip=0, cap=0):
    g[str(i)] = {"class_type": "VHS_LoadVideo", "inputs": dict(
        video=video, force_rate=0, custom_width=0, custom_height=0, frame_load_cap=cap, skip_first_frames=skip,
        select_every_nth=1, format="Wan")}
    return [str(i), 0]

def build_preprocess(prepped, prefix):
    """Stage A. `prepped` is the clip already at 16 fps and WIDTH×HEIGHT (made with ffmpeg in step 8)."""
    W, H = WIDTH, HEIGHT
    g = {}
    def n(i, cls, **inputs):
        g[str(i)] = {"class_type": cls, "inputs": inputs}
        return [str(i), 0]
    src = _load(g, 1, prepped)
    n(2, "PixelPerfectResolution", original_image=src, image_gen_width=W, image_gen_height=H, resize_mode="Just Resize")
    # pose (body + hands) drives the motion; a second pass with the face finds the face to crop
    n(3, "DWPreprocessor", image=src, detect_hand="enable", detect_body="enable", detect_face="disable", resolution=["2", 0],
      bbox_detector="yolox_l.torchscript.pt", pose_estimator="dw-ll_ucoco_384_bs5.torchscript.pt", scale_stick_for_xinsr_cn="disable")
    n(4, "DWPreprocessor", image=src, detect_hand="disable", detect_body="enable", detect_face="enable", resolution=["2", 0],
      bbox_detector="yolox_l.torchscript.pt", pose_estimator="dw-ll_ucoco_384_bs5.torchscript.pt", scale_stick_for_xinsr_cn="disable")
    n(5, "FaceMaskFromPoseKeypoints", pose_kps=["4", 1], person_index=PERSON_INDEX)
    n(6, "ImageCropByMaskAndResize", image=src, mask=["5", 0], base_resolution=512, padding=0,
      min_crop_resolution=128, max_crop_resolution=512)
    # the person to replace: SAM 2 follows them through the clip from the points picked in step 5
    n(7, "DownloadAndLoadSAM2Model", model=SAM2_MODEL, segmentor="video", device="cuda", precision="fp16")
    n(8, "Sam2Segmentation", sam2_model=["7", 0], image=src, keep_model_loaded=False,
      coordinates_positive=json.dumps([{"x": x, "y": y} for x, y in POINTS_XY]), individual_objects=False)
    n(9, "GrowMask", mask=["8", 0], expand=MASK_GROW, tapered_corners=True)
    n(10, "BlockifyMask", masks=["9", 0], block_size=32, device="cpu")
    n(11, "DrawMaskOnImage", image=src, mask=["10", 0], color="0, 0, 0", device="cpu")   # the background with the person blanked
    n(12, "MaskToImage", mask=["10", 0])
    _save(g, 20, ["3", 0], f"{prefix}_pose")
    _save(g, 21, ["6", 0], f"{prefix}_face")
    _save(g, 22, ["11", 0], f"{prefix}_background")
    _save(g, 23, ["12", 0], f"{prefix}_mask", crf=0)
    return g

def build_chunk(ref_image, parts, start, chunk, continue_video=None, prefix="chunk_00"):
    """Stage B: one chunk from frames [start, start + chunk) of the stage-A videos."""
    W, H = WIDTH, HEIGHT
    g = {}
    def n(i, cls, **inputs):
        g[str(i)] = {"class_type": cls, "inputs": inputs}
        return [str(i), 0]
    n(1, "UnetLoaderGGUF", unet_name=UNET_FILE)
    n(2, "LoraLoaderModelOnly", model=["1", 0], lora_name=LIGHTX2V_FILE, strength_model=1.0)
    n(3, "LoraLoaderModelOnly", model=["2", 0], lora_name=RELIGHT_FILE, strength_model=RELIGHT_STRENGTH)
    n(4, "ModelSamplingSD3", model=["3", 0], shift=8.0)
    n(5, "CLIPLoaderGGUF", clip_name=TEXT_ENCODER_FILE, type="wan")
    n(6, "VAELoader", vae_name=VAE_FILE)
    n(7, "CLIPVisionLoader", clip_name=CLIP_VISION_FILE)
    n(8, "CLIPTextEncode", clip=["5", 0], text=PROMPT)
    n(9, "CLIPTextEncode", clip=["5", 0], text=NEGATIVE)
    n(10, "LoadImage", image=ref_image)
    n(11, "ImageResizeKJv2", image=["10", 0], width=W, height=H, upscale_method="lanczos", keep_proportion="pad_edge_pixel",
      pad_color="0, 0, 0", crop_position="center", divisible_by=16, device="cpu")
    n(12, "CLIPVisionEncode", clip_vision=["7", 0], image=["11", 0], crop="none")
    pose = _load(g, 20, parts["pose"], start, chunk)
    face = _load(g, 21, parts["face"], start, chunk)
    bg = _load(g, 22, parts["background"], start, chunk)
    mimg = _load(g, 23, parts["mask"], start, chunk)
    n(24, "ImageToMask", image=mimg, channel="red")
    extra = {}
    if continue_video:
        extra["continue_motion"] = _load(g, 30, continue_video)
    g["41"] = {"class_type": "WanAnimateToVideo", "inputs": dict(
        positive=["8", 0], negative=["9", 0], vae=["6", 0], width=W, height=H, length=chunk, batch_size=1,
        continue_motion_max_frames=5, video_frame_offset=0, clip_vision_output=["12", 0], reference_image=["11", 0],
        face_video=face, pose_video=pose, background_video=bg, character_mask=["24", 0], **extra)}
    n(42, "KSampler", model=["4", 0], positive=["41", 0], negative=["41", 1], latent_image=["41", 2], seed=SEED,
      steps=STEPS, cfg=1.0, sampler_name="euler", scheduler="simple", denoise=1.0)
    n(43, "TrimVideoLatent", samples=["42", 0], trim_amount=["41", 3])
    n(44, "VAEDecode", samples=["43", 0], vae=["6", 0])
    frames = ["44", 0]
    if continue_video:   # drop the frames that repeat the end of the previous chunk
        frames = n(45, "ImageFromBatch", image=["44", 0], batch_index=["41", 4], length=4096)
    _save(g, 46, frames, prefix, crf=12)
    return g

def chunk_plan(total, chunk, overlap=5):
    """Start frame of each chunk. Chunk 1 makes `chunk` frames; each later one starts 5 frames back (re-using the end of
    the previous chunk for continuity) and adds chunk - 5 new ones."""
    starts, made = [], 0
    while made < total:
        starts.append(0 if not starts else made - overlap)
        made += chunk if len(starts) == 1 else chunk - overlap
    return starts, made
print("workflow ready")

In [ ]:
#@title 7 · Start ComfyUI in the background
import subprocess, time, urllib.request, os, sys
def comfy_up():
    try: urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); return True
    except Exception: return False
def start_comfy():
    if comfy_up(): return True
    log = open("/content/comfyui.log", "a")
    # --cache-none: don't keep every node's output in RAM between runs (the free tier has only 12.7 GB)
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch",
                      "--preview-method", "none", "--cache-none"], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(150):
        if comfy_up(): return True
        time.sleep(2)
    return False
print("✓ ComfyUI is running" if start_comfy() else "✗ ComfyUI did not start — last lines of /content/comfyui.log:\n" + open("/content/comfyui.log").read()[-3000:])

In [ ]:
#@title 8 · Render (pose / face / mask once, then chunk by chunk)
import json, time, shutil, subprocess, urllib.request, urllib.error, os, glob
try: import psutil
except ImportError: psutil = None
API = "http://127.0.0.1:8188"
def call(path, data=None):
    req = urllib.request.Request(API + path, data=json.dumps(data).encode() if data is not None else None,
                                 headers={"Content-Type": "application/json"})
    try: return json.loads(urllib.request.urlopen(req, timeout=30).read() or b"{}")
    except urllib.error.HTTPError as e: raise SystemExit("ComfyUI refused the workflow:\n" + e.read().decode()[:3000])
def crashed(what):
    tail = open("/content/comfyui.log").read()[-2500:]
    raise SystemExit(f"ComfyUI stopped during {what} — almost always the free tier running out of RAM.\n"
                     "Try: QUANT = Q3_K_M in step 2 (run 2, 3, then 8 again), or a shorter SECONDS / CHUNK_FRAMES 49.\n"
                     "Step 8 restarts ComfyUI by itself and skips the parts already done.\n\n--- last lines of the log ---\n" + tail)
def run(graph, what):
    if not start_comfy(): crashed("start-up")
    try: pid = call("/prompt", {"prompt": graph})["prompt_id"]
    except urllib.error.URLError: crashed(what)
    t0 = time.time()
    while True:
        try: h = call(f"/history/{pid}")
        except (urllib.error.URLError, ConnectionError, TimeoutError): crashed(what)
        if pid in h:
            st = h[pid]["status"]
            if st.get("status_str") == "error":
                err = next((m[1] for m in st["messages"] if m[0] == "execution_error"), {})
                raise SystemExit(f"{what} failed in {err.get('node_type')}: {err.get('exception_message')}\nSee /content/comfyui.log")
            break
        ram = f" · RAM {psutil.virtual_memory().used / 1e9:.1f}/{psutil.virtual_memory().total / 1e9:.1f} GB" if psutil else ""
        print(f"\r{what} · {int(time.time() - t0) // 60}:{int(time.time() - t0) % 60:02d}{ram}   ", end="")
        time.sleep(5)
    print(f"\r✓ {what} in {(time.time() - t0) / 60:.1f} min" + " " * 30)
def newest(prefix):
    f = [v for v in sorted(glob.glob(f"{ROOT}/output/{prefix}_*.mp4")) if not v.endswith("-audio.mp4")]
    return f[-1] if f else None

# what to make: the clip at 16 fps, cropped to WIDTH×HEIGHT
dur = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", f"{ROOT}/input/{VIDEO}"],
                           capture_output=True, text=True).stdout or 0)
TOTAL = max(9, min(int((dur - START_AT_SECONDS) * 16), int(SECONDS * 16)))
chunk = min(CHUNK_FRAMES, max(9, (TOTAL + 2) // 4 * 4 + 1))   # one chunk if it fits
starts, planned = chunk_plan(TOTAL, chunk)
print(f"{TOTAL} frames ({TOTAL / 16:.1f}s) → {len(starts)} chunk(s) of {chunk} frames")

# a run is identified by its inputs, so re-running step 8 after a crash picks up where it stopped
key = f"{VIDEO}|{REF}|{WIDTH}x{HEIGHT}|{START_AT_SECONDS}|{planned}|{POINTS_XY}|{MASK_GROW}|{chunk}|{PROMPT}|{SEED}|{STEPS}|{QUANT}|{RELIGHT_STRENGTH}"
import hashlib
RUN = hashlib.md5(key.encode()).hexdigest()[:8]
prep_key = hashlib.md5(f"{VIDEO}|{WIDTH}x{HEIGHT}|{START_AT_SECONDS}|{planned}|{POINTS_XY}|{MASK_GROW}".encode()).hexdigest()[:8]
os.makedirs(f"{ROOT}/output", exist_ok=True)

prepped = f"prepped_{prep_key}.mp4"
if not os.path.exists(f"{ROOT}/input/{prepped}"):
    vf = (f"fps=16,scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT},"
          f"tpad=stop_mode=clone:stop_duration=30")
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-ss", str(START_AT_SECONDS), "-i", f"{ROOT}/input/{VIDEO}", "-vf", vf,
                    "-frames:v", str(planned), "-an", "-c:v", "libx264", "-crf", "10", "-pix_fmt", "yuv420p",
                    f"{ROOT}/input/{prepped}"], check=True)

# stage A: pose, face, the person's mask and the background, saved as videos
A = f"prep_{prep_key}"
names = ("pose", "face", "background", "mask")
if not all(newest(f"{A}_{p}") for p in names):
    run(build_preprocess(prepped, A), "pose, face and mask")
parts = {}
for p in names:
    parts[p] = f"{A}_{p}.mp4"; shutil.copy(newest(f"{A}_{p}"), f"{ROOT}/input/{parts[p]}")
call("/free", {"unload_models": True, "free_memory": True})
print("Check these if the result looks wrong:", newest(f"{A}_mask"), "(white = replaced) and", newest(f"{A}_pose"))

# stage B: the chunks
CHUNKS, prev = [], None
for k, s in enumerate(starts):
    prefix = f"run{RUN}_chunk_{k:02d}"
    if not newest(prefix):
        g = build_chunk(REF, parts, s, chunk, continue_video=prev, prefix=prefix)
        json.dump({"prompt": g}, open(f"/content/workflow_api_chunk{k}.json", "w"), indent=1)
        run(g, f"chunk {k + 1}/{len(starts)}")
    else:
        print(f"✓ chunk {k + 1}/{len(starts)} already done")
    CHUNKS.append(newest(prefix))
    prev = f"prev_{prefix}.mp4"; shutil.copy(CHUNKS[-1], f"{ROOT}/input/{prev}")
print("✓ all chunks rendered")

In [ ]:
#@title 9 · Join, add the original sound, preview and download
import subprocess, os
from IPython.display import HTML, display
from base64 import b64encode
lst = "/content/chunks.txt"
open(lst, "w").write("".join(f"file '{c}'\n" for c in CHUNKS))
joined = f"/content/replaced_{RUN}_16fps.mp4"
subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-f", "concat", "-safe", "0", "-i", lst,
                "-ss", str(START_AT_SECONDS), "-i", f"{ROOT}/input/{VIDEO}",
                "-map", "0:v", "-map", "1:a?", "-t", f"{TOTAL / 16:.3f}", "-c:v", "libx264", "-crf", "14", "-pix_fmt", "yuv420p",
                "-c:a", "aac", "-b:a", "192k", "-shortest", joined], check=True)
final = joined
if SMOOTH_30FPS_1080P:
    final = f"/content/replaced_{RUN}_30fps.mp4"
    long_side = "scale=-2:1920" if HEIGHT >= WIDTH else "scale=1920:-2"
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", joined, "-vf",
                    f"minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:vsbmc=1,{long_side}:flags=lanczos,unsharp=5:5:0.4",
                    "-c:v", "libx264", "-crf", "16", "-preset", "slow", "-pix_fmt", "yuv420p", "-c:a", "copy", final], check=True)
print("✓", final, f"({os.path.getsize(final) / 1e6:.1f} MB)")
display(HTML(f'<video width="{min(WIDTH, 420)}" controls src="data:video/mp4;base64,{b64encode(open(final, "rb").read()).decode()}"></video>'))
from google.colab import files
files.download(final)

### If something goes wrong

- **"ComfyUI stopped during …" (out of RAM):** set QUANT to `Q3_K_M` in step 2, run steps 2 and 3, then step 8 again.
  Step 8 restarts ComfyUI itself and skips whatever already finished. A shorter `CHUNK_FRAMES` (49) also helps.
- **Bits of the original person show:** raise `MASK_GROW` (30–40), or add more points on the person in step 5. Check
  `prep_*_mask.mp4` in `/content/ComfyUI/output`: the person should be a white block in every frame.
- **The wrong person was replaced:** move the points in step 5 onto the right one.
- **The face looks off:** use a character picture with a clear, front-facing face at a similar angle to the video.
- **Jitter between chunks:** keep the camera steady in the source video; a longer `CHUNK_FRAMES` (81) means fewer joins.
- **Black or noisy frames:** change `SEED`, or try `STEPS` 8.
- **Too slow:** fewer `SECONDS`, or 416×736 (vertical) in step 2.

### Open the ComfyUI interface (optional)

The steps above don't need it. If you'd like to see or tweak the graph, run the cell below: it gives you a public link
to this ComfyUI. Drag `/content/workflow_api_chunk0.json` (saved by step 8) onto the canvas to load the workflow.
The free Colab tier sometimes warns about web interfaces; the headless steps above avoid that.

In [ ]:
#@title 10 · (optional) Public link to the ComfyUI interface
import subprocess, re, time, os
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True, check=True)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: print("ComfyUI:", m.group(0), "(works while this notebook runs; anyone with the link can use it)"); break